# Rail Corrugation
## Feature Engineering

This notebook converts each raw recording into a compact set of time-domain, side-specific and frequency-domain features for classification.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import welch
from scipy.stats import kurtosis, skew

DATA_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "Train"
LABELS_PATH = DATA_DIR / "Train_Labels.csv"

FS = 10000

labels = pd.read_csv(LABELS_PATH)

In [2]:
sample = pd.read_csv(TRAIN_PATH / "Train1.csv")

vib_cols = [c for c in sample.columns if "vibration" in c.lower()]
shock_cols = [c for c in sample.columns if "shock" in c.lower()]

side1_vib = [c for c in vib_cols if any(f"position {p}" in c.lower() for p in [1, 3, 5, 7])]
side2_vib = [c for c in vib_cols if any(f"position {p}" in c.lower() for p in [2, 4, 6, 8])]

In [3]:
def signal_features(x):
    x = np.asarray(x)

    return {
        "rms": np.sqrt(np.mean(x ** 2)),
        "std": np.std(x),
        "mean_abs": np.mean(np.abs(x)),
        "peak": np.max(np.abs(x)),
        "kurtosis": kurtosis(x),
        "skew": skew(x)
    }

signal_features(sample[vib_cols].to_numpy().ravel())

{'rms': np.float64(0.18817565732472016),
 'std': np.float64(0.18319163126182797),
 'mean_abs': np.float64(0.13654668807983397),
 'peak': np.float64(1.4892578125),
 'kurtosis': np.float64(3.371025927266216),
 'skew': np.float64(0.2579917155414833)}

In [4]:
side1 = signal_features(sample[side1_vib].to_numpy().ravel())
side2 = signal_features(sample[side2_vib].to_numpy().ravel())

side1, side2

({'rms': np.float64(0.19692817591439166),
  'std': np.float64(0.19093758543460196),
  'mean_abs': np.float64(0.1438631820678711),
  'peak': np.float64(1.4892578125),
  'kurtosis': np.float64(3.4256713500946745),
  'skew': np.float64(0.2339455727513847)},
 {'rms': np.float64(0.17899566908227538),
  'std': np.float64(0.17494999264592967),
  'mean_abs': np.float64(0.12923019409179687),
  'peak': np.float64(1.33056640625),
  'kurtosis': np.float64(3.189972038019392),
  'skew': np.float64(0.30525218894876266)})

In [5]:
side1_rms = side1["rms"]
side2_rms = side2["rms"]

print("Difference:", side1_rms - side2_rms)
print("Ratio:", side1_rms / side2_rms)

Difference: 0.01793250683211628
Ratio: 1.1001840263736973


In [6]:
signal = sample[side1_vib].mean(axis=1).to_numpy()

f, psd = welch(signal, fs=FS, nperseg=2048)

bands = [
    (0, 100),
    (100, 250),
    (250, 500),
    (500, 1000),
    (1000, 2000),
    (2000, 5000)
]

In [7]:
for low, high in bands:
    mask = (f >= low) & (f < high)
    power = np.trapezoid(psd[mask], f[mask])

    print(f"{low}-{high} Hz:", power)

def spectral_features(x):
    f, psd = welch(np.asarray(x), fs=FS, nperseg=2048)

    features = {"dominant_freq": f[np.argmax(psd)]}

    for low, high in bands:
        mask = (f >= low) & (f < high)
        features[f"power_{low}_{high}"] = np.trapezoid(psd[mask], f[mask])

    return features

spectral_features(signal)

0-100 Hz: 0.0006827302618787741
100-250 Hz: 0.00034044643528348764
250-500 Hz: 9.252325864410897e-05
500-1000 Hz: 7.008106891886228e-06
1000-2000 Hz: 2.372647242752073e-06
2000-5000 Hz: 3.408559013921039e-06


{'dominant_freq': np.float64(53.7109375),
 'power_0_100': np.float64(0.0006827302618787741),
 'power_100_250': np.float64(0.00034044643528348764),
 'power_250_500': np.float64(9.252325864410897e-05),
 'power_500_1000': np.float64(7.008106891886228e-06),
 'power_1000_2000': np.float64(2.372647242752073e-06),
 'power_2000_5000': np.float64(3.408559013921039e-06)}

In [8]:
def extract_features(data):
    s1 = data[side1_vib].to_numpy().ravel()
    s2 = data[side2_vib].to_numpy().ravel()

    s1_time = signal_features(s1)
    s2_time = signal_features(s2)

    s1_spec = spectral_features(data[side1_vib].mean(axis=1).to_numpy())
    s2_spec = spectral_features(data[side2_vib].mean(axis=1).to_numpy())

    features = {}

    features.update({f"side1_{k}": v for k, v in s1_time.items()})
    features.update({f"side2_{k}": v for k, v in s2_time.items()})
    features.update({f"side1_{k}": v for k, v in s1_spec.items()})
    features.update({f"side2_{k}": v for k, v in s2_spec.items()})

    features["rms_diff"] = s1_time["rms"] - s2_time["rms"]
    features["rms_ratio"] = s1_time["rms"] / s2_time["rms"]

    return features

rows = []

for _, row in labels.iterrows():
    data = pd.read_csv(TRAIN_PATH / row["filename"])

    features = extract_features(data)
    features["filename"] = row["filename"]
    features["label"] = row["label"]

    rows.append(features)

feature_df = pd.DataFrame(rows)

In [9]:
print("Feature matrix:", feature_df.shape)

display(feature_df.head())

Feature matrix: (272, 30)


,side1_rms,side1_std,side1_mean_abs,side1_peak,side1_kurtosis,side1_skew,side2_rms,side2_std,side2_mean_abs,side2_peak,...,side2_power_0_100,side2_power_100_250,side2_power_250_500,side2_power_500_1000,side2_power_1000_2000,side2_power_2000_5000,rms_diff,rms_ratio,filename,label
0,0.196928,0.190938,0.143863,1.489258,3.425671,0.233946,0.178996,0.174950,0.129230,1.330566,...,0.000579,0.000313,0.000064,0.000007,0.000002,0.000004,0.017933,1.100184,Train1.csv,Normal
1,0.916845,0.915718,0.488347,19.201660,38.651499,1.865268,0.970224,0.969623,0.494209,26.440430,...,0.016843,0.006409,0.002893,0.002588,0.000261,0.000080,-0.053379,0.944982,Train2.csv,Side II
2,0.489453,0.487063,0.322309,10.131836,24.345089,0.395546,0.505371,0.503937,0.333544,8.630371,...,0.003987,0.001230,0.001336,0.002106,0.000155,0.000022,-0.015918,0.968503,Train3.csv,Normal
3,0.417142,0.414343,0.267033,6.518555,14.187037,0.413664,0.534819,0.533455,0.307011,13.818359,...,0.005535,0.001433,0.000591,0.000665,0.000047,0.000030,-0.117678,0.779967,Train4.csv,Normal
4,0.076755,0.059765,0.067120,0.378418,0.598831,0.605512,0.060023,0.046401,0.048291,0.366211,...,0.000011,0.000002,0.000015,0.000003,0.000002,0.000004,0.016732,1.278760,Train5.csv,Normal


In [10]:
print("NaN:", feature_df.isna().sum().sum())
print("Infinite:", np.isinf(feature_df.select_dtypes("number")).sum().sum())

NaN: 0
Infinite: 0


In [11]:
display(
    feature_df
    .groupby("label")
    .mean(numeric_only=True)
    .T
)

label,Normal,Side I,Side II
side1_rms,0.303036,0.616152,0.635046
side1_std,0.296513,0.614335,0.633173
side1_mean_abs,0.197587,0.350589,0.409343
side1_peak,4.949117,10.723005,8.203634
side1_kurtosis,19.807360,31.107667,13.246201
side1_skew,0.599593,0.950679,0.240914
side2_rms,0.305121,0.536200,0.803052
side2_std,0.300547,0.534848,0.802219
side2_mean_abs,0.195233,0.329697,0.490485
side2_peak,5.109269,8.052281,9.431966


In [12]:
OUTPUT_PATH = DATA_DIR / "rail_features.csv"
feature_df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)

Saved: ../rail_features.csv
